In [1]:
#import required Libaries
import os
import tensorflow as tf
from tensorflow.keras import layers, regularizers, models

In [2]:
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("marquis03/bdd100k-scenario-classification")

print("Path to dataset files:", path)

Path to dataset files: /root/.cache/kagglehub/datasets/marquis03/bdd100k-scenario-classification/versions/1


In [4]:
print(os.listdir(path))

['val', 'test', 'train']


In [5]:
train_path = os.path.join(path, "train")
print(os.listdir(train_path))

['gas stations', 'tunnel', 'city street', 'highway', 'unknown', 'residential', 'parking lot']


In [6]:
# loading and spliting the dataset for train/test/validation

img_size = (224, 224)
batch_size = 32

train_dataset = tf.keras.utils.image_dataset_from_directory(
    os.path.join(path, "train"),
    validation_split=0.3,
    subset="training",
    seed=42,
    image_size=img_size,
    batch_size=batch_size
)
test_dataset = tf.keras.utils.image_dataset_from_directory(
    os.path.join(path, "train"),
    validation_split=0.3,
    subset="validation",
    seed=42,
    image_size=img_size,
    batch_size=batch_size
)
val_dataset = tf.keras.utils.image_dataset_from_directory(
    os.path.join(path, "val"),
    image_size=img_size,
    batch_size=batch_size
)



Found 69863 files belonging to 7 classes.
Using 48905 files for training.
Found 69863 files belonging to 7 classes.
Using 20958 files for validation.
Found 10000 files belonging to 7 classes.


In [7]:
# Preprocess the data
normalization_layer = tf.keras.layers.Rescaling(1./255)

train_dataset = train_dataset.map(lambda x, y: (normalization_layer(x), y))
test_dataset  = test_dataset.map(lambda x, y: (normalization_layer(x), y))
val_dataset   = val_dataset.map(lambda x, y: (normalization_layer(x), y))

In [8]:
# Build Resnet Block
class ResidualBlock(tf.keras.layers.Layer):
    def __init__(self, filters, stride=1):
        super().__init__()

        self.conv1 = layers.Conv2D(filters, 3, strides=stride, padding='same')
        self.bn1 = layers.BatchNormalization()

        self.conv2 = layers.Conv2D(filters, 3, strides=1, padding='same')
        self.bn2 = layers.BatchNormalization()


        if stride != 1:
            self.shortcut = tf.keras.Sequential([
                layers.Conv2D(filters, 1, strides=stride, padding='same'),
                layers.BatchNormalization()
            ])
        else:
            self.shortcut = lambda x: x   # identity

    def call(self, x):
        shortcut = self.shortcut(x)

        x = tf.nn.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))

        x = x + shortcut
        return tf.nn.relu(x)

In [9]:
# Build Resnet Model
def build_resnet(input_shape=(224,224,3), num_classes=7):

    inputs = layers.Input(shape=input_shape)

    # Initial layer
    x = layers.Conv2D(64, 7, strides=2, padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D(pool_size=3, strides=2, padding='same')(x)

    # Residual stages
    # Stage 1
    x = ResidualBlock(64, stride=1)(x)
    x = ResidualBlock(64, stride=1)(x)

    # Stage 2
    x = ResidualBlock(128, stride=2)(x)
    x = ResidualBlock(128, stride=1)(x)

    # Stage 3
    x = ResidualBlock(256, stride=2)(x)
    x = ResidualBlock(256, stride=1)(x)

    # Stage 4
    x = ResidualBlock(512, stride=2)(x)
    x = ResidualBlock(512, stride=1)(x)

    # Classification head
    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dense(
        128,
        activation='relu',
        kernel_regularizer=regularizers.l2(0.001)
    )(x)

    x = layers.Dropout(0.5)(x)

    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = models.Model(inputs, outputs)

    return model

In [10]:
# Configuring the model howto learn
model = build_resnet()

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 112, 112, 64)   │         9,472 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ residual_block (ResidualBlock)  │ (None, 56, 56, 64)     │        74,368 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ residual_block_1                │ (None, 56, 56, 64)     │        74,368 │
│ (ResidualBlock)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ residual_block_2                │ (None, 28, 28, 128)    │       231,296 │
│ (ResidualBlock)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ residual_block_3                │ (None, 28, 28, 128)    │       296,192 │
│ (ResidualBlock)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ residual_block_4                │ (None, 14, 14, 256)    │       921,344 │
│ (ResidualBlock)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ residual_block_5                │ (None, 14, 14, 256)    │     1,182,208 │
│ (ResidualBlock)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ residual_block_6                │ (None, 7, 7, 512)      │     3,677,696 │
│ (ResidualBlock)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ residual_block_7                │ (None, 7, 7, 512)      │     4,723,712 │
│ (ResidualBlock)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 512)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        65,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,257,479 (42.94 MB)

 Trainable params: 11,247,879 (42.91 MB)

 Non-trainable params: 9,600 (37.50 KB)

In [11]:
# Train the model
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=10
)

Epoch 1/10
1529/1529 ━━━━━━━━━━━━━━━━━━━━ 326s 210ms/step - accuracy: 0.6608 - loss: 1.0558 - val_accuracy: 0.6703 - val_loss: 1.0236
Epoch 2/10
1529/1529 ━━━━━━━━━━━━━━━━━━━━ 322s 210ms/step - accuracy: 0.6990 - loss: 0.9085 - val_accuracy: 0.6653 - val_loss: 1.4439
Epoch 3/10
1529/1529 ━━━━━━━━━━━━━━━━━━━━ 321s 210ms/step - accuracy: 0.7126 - loss: 0.8314 - val_accuracy: 0.6242 - val_loss: 0.9689
Epoch 4/10
1529/1529 ━━━━━━━━━━━━━━━━━━━━ 321s 210ms/step - accuracy: 0.7269 - loss: 0.7713 - val_accuracy: 0.7016 - val_loss: 0.8142
Epoch 5/10
1529/1529 ━━━━━━━━━━━━━━━━━━━━ 321s 210ms/step - accuracy: 0.7394 - loss: 0.7120 - val_accuracy: 0.6160 - val_loss: 1.0135
Epoch 6/10
1529/1529 ━━━━━━━━━━━━━━━━━━━━ 321s 210ms/step - accuracy: 0.7598 - loss: 0.6474 - val_accuracy: 0.6225 - val_loss: 1.0743
Epoch 7/10
1529/1529 ━━━━━━━━━━━━━━━━━━━━ 321s 210ms/step - accuracy: 0.7935 - loss: 0.5595 - val_accuracy: 0.6556 - val_loss: 1.2890
Epoch 8/10
1529/1529 ━━━━━━━━━━━━━━━━━━━━ 321s 210ms/step - ac

In [12]:
# Evaluate model performance
test_loss, test_acc = model.evaluate(val_dataset)
print("Test Accuracy:", test_acc)

313/313 ━━━━━━━━━━━━━━━━━━━━ 13s 41ms/step - accuracy: 0.6337 - loss: 1.5573
Test Accuracy: 0.6337000131607056
